In [76]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

import joblib

In [77]:
df = pd.read_csv(
    "behavior_feature_engineered_data.csv"
)

In [78]:
joblib.dump(

    X.columns.tolist(),

    "feature_columns.pkl"
)

['feature_columns.pkl']

In [79]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

In [80]:
print(X.dtypes)

user_id                       int64
country                       int64
device_type                   int64
browser                       int64
failed_attempts               int64
login_status                  int64
hour                          int64
day_of_week                   int64
is_weekend                    int64
max_normal_failed_attempts    int64
usual_start_hour              int64
usual_end_hour                int64
is_new_country                int64
is_new_device                 int64
is_new_browser                int64
unusual_login_hour            int64
excessive_failed_attempts     int64
behavior_risk_score           int64
high_risk_login               int64
dtype: object


In [81]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [82]:
iso_model = IsolationForest(

    n_estimators=200,

    contamination=0.05,

    max_samples='auto',

    random_state=42
)

iso_model.fit(X_train_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [83]:
preds = iso_model.predict(X_test_scaled)

preds = np.where(
    preds == -1,
    1,
    0
)

In [84]:
print(confusion_matrix(y_test, preds))

print("\n")

print(classification_report(y_test, preds))

[[1900    0]
 [   0  100]]


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1900
           1       1.00      1.00      1.00       100

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [85]:
scores = iso_model.decision_function(
    X_test_scaled
)

In [86]:
risk_scores = -scores

In [87]:
from sklearn.preprocessing import MinMaxScaler

risk_scaler = MinMaxScaler(
    feature_range=(0, 100)
)

normalized_risk_scores = risk_scaler.fit_transform(

    risk_scores.reshape(-1, 1)

).flatten()

In [88]:
results = X_test.copy()

results["actual_label"] = y_test.values

results["predicted_label"] = preds

results["risk_score"] = normalized_risk_scores

In [89]:
results.sort_values(
    by="risk_score",
    ascending=False
).head(10)

,user_id,country,device_type,browser,failed_attempts,login_status,hour,day_of_week,is_weekend,max_normal_failed_attempts,...,is_new_country,is_new_device,is_new_browser,unusual_login_hour,excessive_failed_attempts,behavior_risk_score,high_risk_login,actual_label,predicted_label,risk_score
3042,30,2,0,3,20,0,3,6,1,20,...,1,1,1,0,0,3,1,1,1,100.000000
2426,24,0,1,3,19,0,2,6,1,19,...,1,1,1,0,0,3,1,1,1,98.148654
3022,30,5,2,3,14,0,2,5,1,20,...,1,1,1,0,0,3,1,1,1,97.992727
1486,14,4,4,3,19,0,0,5,1,20,...,1,1,1,0,0,3,1,1,1,97.551326
3026,30,1,2,2,16,0,1,6,1,20,...,1,1,1,0,0,3,1,1,1,96.110316
8085,80,5,1,1,9,0,23,0,0,9,...,1,1,1,0,0,3,1,1,1,95.477490
9964,99,2,0,2,13,0,23,6,1,17,...,1,1,1,0,0,3,1,1,1,95.457004
7002,70,0,0,1,18,0,4,1,0,18,...,1,1,1,0,0,3,1,1,1,95.317588
8751,87,5,2,0,9,0,23,2,0,20,...,1,1,1,0,0,3,1,1,1,95.300282
2685,26,5,2,1,20,0,23,3,0,20,...,1,1,1,0,0,3,1,1,1,95.219108


In [90]:
def risk_level(score):

    if score >= 80:
        return "Critical"

    elif score >= 60:
        return "High"

    elif score >= 40:
        return "Medium"

    else:
        return "Low"

In [91]:
results["risk_level"] = results[
    "risk_score"
].apply(risk_level)

In [92]:


results[[
    "actual_label",

    "predicted_label",

    "risk_score",

    "risk_level"
]].head(20)

,actual_label,predicted_label,risk_score,risk_level
1225,0,0,19.867946,Low
5819,0,0,27.626220,Low
9733,0,0,38.368777,Low
6399,0,0,12.584046,Low
3521,0,0,4.045391,Low
7552,0,0,5.214483,Low
272,0,0,21.153023,Low
553,0,0,31.520029,Low
1078,0,0,36.341618,Low
9456,0,0,39.842062,Low


In [93]:
joblib.dump(
    iso_model,

    "isolation_forest_model.pkl"
)

['isolation_forest_model.pkl']

In [94]:
joblib.dump(
    scaler,

    "scaler.pkl"
)

['scaler.pkl']

In [95]:
results.to_csv(

    "isolation_forest_results.csv",

    index=False
)

print("Optimization Complete!")

Optimization Complete!
